# Production health requested model contract v1

目标：验证 production health 可以把 dashboard export 中的 requested QuantBench 模型契约作为可选必需 gate。

In [1]:
from __future__ import annotations

import json
import os
import shutil
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT_DIR = ROOT / 'research' / 'production_health_requested_model_contract_v1'
CARD = EXPERIMENT_DIR / 'paper_requested_contract_strategy'
DASHBOARD_EXPORT = ROOT / 'research' / 'dashboard_requested_model_contract_v1' / 'scheduled_dashboard_export.json'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CARD.mkdir(parents=True, exist_ok=True)
assert DASHBOARD_EXPORT.exists(), DASHBOARD_EXPORT
ROOT

PosixPath('/Users/rziiiii/dev/workspace')

In [2]:
from src.core import DatasetSnapshot, publish_production_manifest

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')

dashboard_payload = json.loads(DASHBOARD_EXPORT.read_text(encoding='utf-8'))
contract = dashboard_payload['requested_model_contract']
assert contract['all_requested_models_completed'] is True

shutil.copy2(DASHBOARD_EXPORT, CARD / 'dashboard_export.json')
write_json(CARD / 'production_summary.json', {'status': 'paper_requested_contract_health_completed'})
write_json(CARD / 'intent_ledger.json', {'schema_version': 'api.intent_ledger.v1', 'mode': 'paper'})
write_json(CARD / 'paper_execution_result.json', {'schema_version': 'schedule.paper_execution.v1', 'status': 'success'})
write_json(CARD / 'data' / 'data_registry.json', {'schema_version': 'data.registry.v1'})
write_json(CARD / 'matrix_card' / 'analysis_manifest.json', {'schema_version': 'artifact.v1'})
placeholder_notebook = {
    'cells': [
        {
            'cell_type': 'markdown',
            'metadata': {},
            'source': ['# Paper requested contract strategy\n', '\n', 'production health gate 的占位策略证据。'],
        },
        {
            'cell_type': 'markdown',
            'metadata': {},
            'source': [
                '## 中文结论\n',
                '\n',
                '- 做了什么：构造 paper/sim production manifest，用于验证 production health requested model contract gate。\n',
                '- 核心指标：requested_model_count=32，completed=32，missing=0，incomplete=0。\n',
                '- regime 差异：本占位策略不重新计算 regime，只引用 dashboard export 的 analysis 结果。\n',
                '- 是否晋升：不晋升。该 notebook 仅作为 production health gate 证据。\n',
                '- 风险：不代表实盘策略收益，只验证 manifest 与 dashboard contract 健康检查。\n',
                '- 下一步：真实 production 策略需要使用自己的 notebook 和固定参数发布。',
            ],
        },
    ],
    'metadata': {'language_info': {'name': 'python'}},
    'nbformat': 4,
    'nbformat_minor': 5,
}
(CARD / 'paper_requested_contract_strategy.ipynb').write_text(
    json.dumps(placeholder_notebook, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

dataset = DatasetSnapshot(
    name='paper_requested_contract_health',
    universe=('SOXL',),
    start='2026-01-01',
    end='2026-06-09',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=('data/data_registry.json',),
)
manifest = publish_production_manifest(
    strategy_id='paper_requested_contract_strategy',
    output_dir=CARD,
    registry_path=EXPERIMENT_DIR / 'production_registry.json',
    notebook_path='paper_requested_contract_strategy.ipynb',
    dataset=dataset,
    paths={
        'summary': 'production_summary.json',
        'dashboard_export': 'dashboard_export.json',
        'intent_ledger': 'intent_ledger.json',
        'paper_execution_result': 'paper_execution_result.json',
        'data_registry': 'data/data_registry.json',
        'analysis_manifest': 'matrix_card/analysis_manifest.json',
    },
    metadata={
        'status': 'paper_requested_contract_health_completed',
        'mode': 'paper',
        'live_broker_enabled': False,
        'paper_execution_pipeline_schema_version': 'schedule.paper_execution.v1',
        'paper_execution_job_status': 'success',
        'requested_model_contract_required': True,
    },
    published_at='2026-06-19T12:40:00Z',
)

manifest['manifest_hash']

'1507fc0e58b0cac8e7e291b1a21e39905d8bae00968d849d0efe7b2f38c24036'

In [3]:
from src.schedule import ProductionHealthCheckConfig, run_production_health_check_once

events: list[str] = []
config = ProductionHealthCheckConfig(
    name='production_requested_contract_health',
    registry_path=EXPERIMENT_DIR / 'production_registry.json',
    strategy_id='paper_requested_contract_strategy',
    output_path=EXPERIMENT_DIR / 'production_health.json',
    require_requested_model_contract=True,
    notify_details=True,
)

result = run_production_health_check_once(config, lock_dir=EXPERIMENT_DIR / 'locks', notify=events.append)
health = result.payload

assert result.status == 'success'
assert health['health_status'] == 'healthy'
assert health['is_healthy'] is True
assert health['requested_model_contract_required'] is True
assert health['requested_model_contract_status'] == 'complete'
assert health['requested_model_count'] == 32
assert health['requested_model_completed_count'] == 32
assert health['requested_model_missing_count'] == 0
assert health['requested_model_incomplete_count'] == 0
assert health['requested_model_all_completed'] is True
assert events and 'requested_models=32/32 missing=0 incomplete=0' in events[0]

health

{'schema_version': 'schedule.production_health.v1',
 'strategy_id': 'paper_requested_contract_strategy',
 'health_status': 'healthy',
 'is_healthy': True,
 'registry_path': '/Users/rziiiii/dev/workspace/research/production_health_requested_model_contract_v1/production_registry.json',
 'manifest_path': '/Users/rziiiii/dev/workspace/research/production_health_requested_model_contract_v1/paper_requested_contract_strategy/production_manifest.json',
 'manifest_hash': '1507fc0e58b0cac8e7e291b1a21e39905d8bae00968d849d0efe7b2f38c24036',
 'registry_manifest_hash': '1507fc0e58b0cac8e7e291b1a21e39905d8bae00968d849d0efe7b2f38c24036',
 'computed_manifest_hash': '1507fc0e58b0cac8e7e291b1a21e39905d8bae00968d849d0efe7b2f38c24036',
 'manifest_hash_matches_registry': True,
 'manifest_hash_matches_content': True,
 'artifact_id': 'paper_requested_contract_strategy',
 'artifact_type': 'production_strategy',
 'mode': 'paper',
 'status': 'paper_requested_contract_health_completed',
 'live_broker_enabled': Fa

In [4]:
summary = {
    'schema_version': 'research.production_health_requested_model_contract.v1',
    'strategy_id': 'paper_requested_contract_strategy',
    'production_manifest_path': str((CARD / 'production_manifest.json').relative_to(ROOT)),
    'production_registry_path': str((EXPERIMENT_DIR / 'production_registry.json').relative_to(ROOT)),
    'production_health_path': str((EXPERIMENT_DIR / 'production_health.json').relative_to(ROOT)),
    'dashboard_export_path': str((CARD / 'dashboard_export.json').relative_to(ROOT)),
    'health_status': health['health_status'],
    'requested_model_contract_status': health['requested_model_contract_status'],
    'requested_model_count': health['requested_model_count'],
    'requested_model_completed_count': health['requested_model_completed_count'],
    'requested_model_missing_count': health['requested_model_missing_count'],
    'requested_model_incomplete_count': health['requested_model_incomplete_count'],
    'notification': events[0],
}
(EXPERIMENT_DIR / 'analysis_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
summary

{'schema_version': 'research.production_health_requested_model_contract.v1',
 'strategy_id': 'paper_requested_contract_strategy',
 'production_manifest_path': 'research/production_health_requested_model_contract_v1/paper_requested_contract_strategy/production_manifest.json',
 'production_registry_path': 'research/production_health_requested_model_contract_v1/production_registry.json',
 'production_health_path': 'research/production_health_requested_model_contract_v1/production_health.json',
 'dashboard_export_path': 'research/production_health_requested_model_contract_v1/paper_requested_contract_strategy/dashboard_export.json',
 'health_status': 'healthy',
 'requested_model_contract_status': 'complete',
 'requested_model_count': 32,
 'requested_model_completed_count': 32,
 'requested_model_missing_count': 0,
 'requested_model_incomplete_count': 0,
 'notification': 'production_requested_contract_health:success | strategy=paper_requested_contract_strategy | health=healthy | missing=0 | p

## 中文结论

- 做了什么：新增 production health 的 requested model contract 可选必需 gate，并用一个 paper/sim production manifest 验证 dashboard export 中的 32 模型契约能被 health check 读取。
- 核心指标：health_status=healthy，requested_model_contract_status=complete，requested_model_count=32，requested_model_completed_count=32，missing=0，incomplete=0。
- regime 差异：本实验不重新比较 regime 收益；它验证生产健康检查可以引用 dashboard export 中已经存在的模型矩阵与 analysis 结果。
- 是否晋升：不晋升生产策略。本实验验证 production health gate，不改变实盘或 paper 执行策略。
- 风险：该 gate 证明 dashboard export 的 requested 模型清单完整；不证明 2026-01-01 至 2026-06-09 短窗口收益可以外推。默认配置保持兼容，只有 `require_requested_model_contract=True` 才阻断缺失或不完整 contract。
- 下一步：生产策略如果要求完整模型矩阵，应在发布 manifest 前使用包含 requested contract 的 dashboard export，并在 health check 中开启该 gate。